# FEATURE ENGINEERING
Input: dataset_eda.csv

Output: dataset_feature_engineered.csv

Goal:
- Transform raw monthly transaction data into behavioral
- indicators ready for K-Means and DBSCAN

In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "dataset_eda.csv"
OUTPUT_PATH = "dataset_feature_engineered.csv"

df = pd.read_csv(DATA_PATH, encoding='latin-1')

# Detect monthly transaction columns: M-35 ... M-0
month_cols = [c for c in df.columns if str(c).startswith("M-")]
month_cols = sorted(month_cols, key=lambda x: int(x.replace("M-", "")), reverse=True)

recent_cols = [c for c in ["M-5", "M-4", "M-3", "M-2", "M-1", "M-0"] if c in df.columns]
past_cols = [c for c in month_cols if c not in recent_cols]

# -----------------------------
# Customer value features
# -----------------------------
df["total_volume"] = df[month_cols].sum(axis=1)
df["avg_monthly_volume"] = df[month_cols].mean(axis=1)

# -----------------------------
# Engagement features
# -----------------------------
df["recent_volume"] = df[recent_cols].sum(axis=1)
df["past_volume"] = df[past_cols].sum(axis=1)
df["recent_ratio"] = df["recent_volume"] / (df["total_volume"] + 1)

# -----------------------------
# Trend features
# -----------------------------
df["volume_trend"] = df["M-0"] - df["M-35"]
df["trend_pct"] = (df["M-0"] - df["M-35"]) / (df["M-35"] + 1)

# -----------------------------
# Stability features
# -----------------------------
df["volatility"] = df[month_cols].std(axis=1)
df["cv_volume"] = df["volatility"] / (df["avg_monthly_volume"] + 1)

# -----------------------------
# Churn-oriented deterioration feature
# -----------------------------
expected_recent = (df["past_volume"] / max(len(past_cols), 1)) * max(len(recent_cols), 1)
df["activity_drop_ratio"] = 1 - (df["recent_volume"] / (expected_recent + 1))

# -----------------------------
# Log transformations
# Important for K-Means/DBSCAN because transaction variables are skewed.
# -----------------------------
for col in ["Revenue", "Employees", "total_volume", "recent_volume", "avg_monthly_volume", "volatility"]:
    if col in df.columns:
        df[f"log_{col.lower()}"] = np.log1p(df[col].clip(lower=0))

df.to_csv(OUTPUT_PATH, index=False)

# print("Feature engineering completed.")
# print("Output saved as:", OUTPUT_PATH)
